# SQL聚合 — COUNT / SUM / AVG / GROUP BY / HAVING（练习）

In [1]:
import duckdb

%load_ext sql
%sql duckdb://

Connecting to 'duckdb://'

In [2]:
# 1. 聚合函数 —— 不分组，对整张表算一个值
# COUNT(*)：总行数
duckdb.sql("SELECT COUNT(*) AS total_orders FROM '../data/sales.csv'")

┌──────────────┐
│ total_orders │
│    int64     │
├──────────────┤
│          500 │
└──────────────┘

In [3]:
# SUM / AVG / MIN / MAX
duckdb.sql("""
    SELECT
        SUM(total) AS total_revenue,
        AVG(total) AS avg_order_value,
        MIN(total) AS min_order,
        MAX(total) AS max_order
    FROM '../data/sales.csv'    
""")

┌───────────────┬─────────────────┬───────────┬───────────┐
│ total_revenue │ avg_order_value │ min_order │ max_order │
│    int128     │     double      │   int64   │   int64   │
├───────────────┼─────────────────┼───────────┼───────────┤
│       1267716 │        2535.432 │        99 │      9995 │
└───────────────┴─────────────────┴───────────┴───────────┘

In [ ]:
# COUNT的三种形态（面试常考区别）：
duckdb.sql("""
    SELECT
        COUNT(*) AS count_all, -- 所有行（包括NULL）
        COUNT(country) AS count_country, -- country非NULL的行数
        COUNT(DISTINCT country) AS count_distinct -- 不重复的country数
    FROM '../data/sales.csv'
""")

# COUNT(DISTINCT xxx)极其常用：“有多少个不重复的客户/产品/国家”

┌───────────┬───────────────┬────────────────┐
│ count_all │ count_country │ count_distinct │
│   int64   │     int64     │     int64      │
├───────────┼───────────────┼────────────────┤
│       500 │           500 │              5 │
└───────────┴───────────────┴────────────────┘

In [5]:
# 2. GROUP BY —— 分组聚合（核心中的核心）
# 每个国家的订单数
duckdb.sql("""
    SELECT country, COUNT(*) AS order_count
    FROM '../data/sales.csv'\
    GROUP BY country 
""")

┌─────────┬─────────────┐
│ country │ order_count │
│ varchar │    int64    │
├─────────┼─────────────┤
│ UK      │         197 │
│ France  │          80 │
│ China   │          32 │
│ Germany │          66 │
│ US      │         125 │
└─────────┴─────────────┘

In [ ]:
# 每个国家的总销售额 + 平均客单价
duckdb.sql("""
    SELECT
        country,
        COUNT(*) AS order_count,
        SUM(total) AS revenue,
        AVG(total) AS avg_value
    FROM '../data/sales.csv'
    GROUP BY country
    ORDER BY revenue DESC 
""")

# 黄金法则：SELECT里出现的列，要么在GROUP BY里，要么被聚合函数包着
# 错误师范：SELECT country, product, SUM(total) ... GROUP BY country
# → product既没分组也没聚合，报错（或返回不确定结果）

┌─────────┬─────────────┬─────────┬───────────────────┐
│ country │ order_count │ revenue │     avg_value     │
│ varchar │    int64    │ int128  │      double       │
├─────────┼─────────────┼─────────┼───────────────────┤
│ UK      │         197 │  534817 │ 2714.807106598985 │
│ US      │         125 │  300613 │          2404.904 │
│ France  │          80 │  208165 │         2602.0625 │
│ Germany │          66 │  144020 │ 2182.121212121212 │
│ China   │          32 │   80101 │        2503.15625 │
└─────────┴─────────────┴─────────┴───────────────────┘

In [8]:
# 3. 多列分组
# 按”国家 + 品类“两个维度分组
duckdb.sql("""
    SELECT 
        country,
        category,
        COUNT(*) AS cnt,
        SUM(total) AS revenue
    FROM '../data/sales.csv'
    GROUP BY country, category
    ORDER BY country, revenue DESC 
""")

┌─────────┬───────────┬───────┬─────────┐
│ country │ category  │  cnt  │ revenue │
│ varchar │  varchar  │ int64 │ int128  │
├─────────┼───────────┼───────┼─────────┤
│ China   │ Computer  │    14 │   32159 │
│ China   │ Accessory │     9 │   21271 │
│ China   │ Mobile    │     7 │   18480 │
│ China   │ Audio     │     2 │    8191 │
│ France  │ Audio     │    17 │   60245 │
│ France  │ Accessory │    23 │   54935 │
│ France  │ Computer  │    23 │   51431 │
│ France  │ Mobile    │    17 │   41554 │
│ Germany │ Computer  │    23 │   51036 │
│ Germany │ Accessory │    22 │   45042 │
│ Germany │ Mobile    │    12 │   28967 │
│ Germany │ Audio     │     9 │   18975 │
│ UK      │ Accessory │    71 │  164595 │
│ UK      │ Computer  │    68 │  158700 │
│ UK      │ Mobile    │    37 │  137788 │
│ UK      │ Audio     │    21 │   73734 │
│ US      │ Accessory │    55 │  155423 │
│ US      │ Computer  │    32 │   70607 │
│ US      │ Audio     │    18 │   43541 │
│ US      │ Mobile    │    20 │   

In [9]:
# 4. HAVING —— 对分组结果过滤
# 找出“订单数超过50的国家”
duckdb.sql("""
    SELECT country, COUNT(*) AS order_count
    FROM '../data/sales.csv'
    GROUP BY country
    HAVING COUNT(*) > 50 
""")

# WHERE vs HAVING —— 面试必考，务必搞清：
# WHERE：在分组【之前】过滤【原始行】 —— 不能用聚合函数
# HAVING：在分组【之后】过滤【分组结果】 —— 可以用聚合函数
# 执行顺序： FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT

┌─────────┬─────────────┐
│ country │ order_count │
│ varchar │    int64    │
├─────────┼─────────────┤
│ UK      │         197 │
│ US      │         125 │
│ France  │          80 │
│ Germany │          66 │
└─────────┴─────────────┘

In [10]:
# 同时用WHERE和HAVING的例子：
duckdb.sql("""
    SELECT country, COUNT(*) AS order_count, SUM(total) AS revenue
    FROM '../data/sales.csv'
    WHERE total > 500 -- 先过滤：只保留金额>500的订单
    GROUP BY country  -- 再分组
    HAVING SUM(total) > 100000 -- 最后过滤：只保留总额>10万的国家
    ORDER BY revenue DESC
""")

┌─────────┬─────────────┬─────────┐
│ country │ order_count │ revenue │
│ varchar │    int64    │ int128  │
├─────────┼─────────────┼─────────┤
│ UK      │         159 │  524119 │
│ US      │          97 │  292588 │
│ France  │          65 │  203512 │
│ Germany │          53 │  139460 │
└─────────┴─────────────┴─────────┘

In [11]:
# 5. 聚合 + 计算
# 聚合结果可以再参与计算
duckdb.sql("""
    SELECT
        category,
        SUM(total) AS revenue,
        COUNT(*) AS orders,
        SUM(total)/COUNT(*) AS revenue_per_order, -- 聚合结果相除
        ROUND(AVG(total), 2) AS avg_rounded -- ROUND 四舍五入
    FROM '../data/sales.csv'
    GROUP BY category
    ORDER BY revenue DESC
""")

┌───────────┬─────────┬────────┬────────────────────┬─────────────┐
│ category  │ revenue │ orders │ revenue_per_order  │ avg_rounded │
│  varchar  │ int128  │ int64  │       double       │   double    │
├───────────┼─────────┼────────┼────────────────────┼─────────────┤
│ Accessory │  441266 │    180 │ 2451.4777777777776 │     2451.48 │
│ Computer  │  363933 │    160 │         2274.58125 │     2274.58 │
│ Mobile    │  257831 │     93 │ 2772.3763440860216 │     2772.38 │
│ Audio     │  204686 │     67 │ 3055.0149253731342 │     3055.01 │
└───────────┴─────────┴────────┴────────────────────┴─────────────┘

In [12]:
# 6. 完整查询的子句顺序（把所有学过的串起来）
# 一个“全要素”查询长这样：
duckdb.sql("""
    SELECT country, category, SUM(total) AS revenue -- 5. 选列/聚合
    FROM '../data/sales.csv'                        -- 1. 数据源
    WHERE total > 200                               -- 2. 行过滤
    GROUP BY country, category                      -- 3. 分组
    HAVING SUM(total) > 20000                       -- 4. 组过滤
    ORDER BY revenue DESC                           -- 6. 排序
    LIMIT 10                                        -- 7. 限量
""")
# 记住这个子句顺序：SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT
# 写错顺序会直接语法报错

┌─────────┬───────────┬─────────┐
│ country │ category  │ revenue │
│ varchar │  varchar  │ int128  │
├─────────┼───────────┼─────────┤
│ UK      │ Accessory │  163704 │
│ UK      │ Computer  │  157314 │
│ US      │ Accessory │  154928 │
│ UK      │ Mobile    │  137788 │
│ UK      │ Audio     │   73635 │
│ US      │ Computer  │   69914 │
│ France  │ Audio     │   60146 │
│ France  │ Accessory │   54737 │
│ France  │ Computer  │   51233 │
│ Germany │ Computer  │   51036 │
└─────────┴───────────┴─────────┘
  10 rows             3 columns